### Import libaries

In [1]:
import os
import sys
import subprocess
import importlib
sys.path.append('DescriPyTor')

import pandas as pd
import numpy as np
from tqdm import tqdm

from datetime import datetime
from pathlib import Path

from desc_utils import atom_numbering as an
from desc_utils import orca_descriptors as od

ROOT_DIR = os.getcwd()
print(f"Current working directory: {ROOT_DIR}")

Current working directory: /Users/julesschleinitz/Desktop/Code/orca_descriptors


### Define the source directory containing the orca and xyz files

Provide path to directory or directory of directories containing xyz files path can be 

1. absolute, e.g. "/home/aconner/orca_descriptors/library_pyox_2026_03_03/",

2. relative, e.g. "library_pyox_2026_03_03" (joined to current working directory).

In [2]:
src_dir = "DATA/LNiH2_library_001_to_download_2026_03_24"  # change this as needed

### Define the atoms to be included in the descriptor.

In [3]:
#Define lists for decriptor retrieval 

# 1 -atom descriptors (e.g. Sterimol L, atomic charge)
atoms = ["H1", "H2", "C1", "C2", "N1", "N2", "Ni"]

# 2-atom descriptors (e.g. Sterimol B1, bond length), if order matters for 2-atom descriptors (e.g. Sterimol B1), atoms must be listed in desired order
bonds = [("C1", "N1"), ("N1", "C1"), ("C2", "N2"), ("N2", "C2"), ("Ni", "N1"), ("Ni", "N2"), ("Ni", "H1"), ("Ni", "H2"), ("C1", "C2"), ("C2", "C1")]
pairs_bending = [("H1", "H2"), ("N1", "C2"), ("N2", "C1")] # list of bonds to consider for bend vibration descriptors, if order matters for bend vibration descriptors, atoms must be listed in desired order

# 3-atom descriptors (e.g. Sterimol B5, bond angle), if order matters for 3-atom descriptors (e.g. Sterimol B5), atoms must be listed in desired order
angles = [("H2", "Ni", "H1"), ("Ni", "N1", "C1"), ("Ni", "N2", "C2"), ("N1", "C1", "C2"), ("N2", "C2", "C1"), ("N1", "Ni", "N2"), ("C1", "Ni", "C2")]

# 4-atom descriptors (e.g. dihedral angle), if order matters for 4-atom descriptors (e.g. dihedral angle), atoms must be listed in desired order
dihedrals = [("N1", "C1", "C2", "N2"), ("C1", "C2", "N2", "Ni"), ("C2", "C1", "N1", "Ni")]

# If you want to include quadrant-based descriptors, define the quadrants here. Each quadrant is a list of atom labels. For example, if you want to define one quadrant that includes atoms C1, N1, C2, N2, and Ni, you would define:
#quadrants = [[2, 1, 3, 4, 6]] 


# Atom Numbering

In [4]:
# example for the numbering of the atoms for one molecule:

df = an.compile_final('DATA/LNiH2_library_001_to_download_2026_03_24/AATATIAROAAKIN-SNVBAGLBSA-N/', subdirectory=True)

In [5]:
# Get numbering for all molecules

# Set your ORCA root folder here (contains many conf_* subfolders)
orca_folder = Path(src_dir)

print(f"Current working directory before processing: {os.getcwd()}")

df_num = pd.DataFrame()
for inchi_folder in os.listdir(orca_folder):
    if os.path.isdir(os.path.join(orca_folder, inchi_folder)):
        inchi_folder_path = os.path.join(orca_folder, inchi_folder)
        print(f"Processing {inchi_folder_path} for atom numbering...")
        df = an.compile_final(inchi_folder_path, subdirectory=True)
        df_num = pd.concat([df_num, df], ignore_index=True)

df_num.head()

# add the feather file paths to the df_num dataframe

df_num['feather_path'] = df_num['path'].apply(lambda x: x + x.split('/')[-2] + '.feather')

df_num.head()

Current working directory before processing: /Users/julesschleinitz/Desktop/Code/orca_descriptors
Processing DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N for atom numbering...
added slash to end of path, new path is DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/
Processing DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N for atom numbering...
added slash to end of path, new path is DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/
Processing DATA/LNiH2_library_001_to_download_2026_03_24/ILENSZIBAAQNCZ-SNVBAGLBSA-N for atom numbering...
added slash to end of path, new path is DATA/LNiH2_library_001_to_download_2026_03_24/ILENSZIBAAQNCZ-SNVBAGLBSA-N/
Processing DATA/LNiH2_library_001_to_download_2026_03_24/YOWGAJZEYGYTFY-SNVBAGLBSA-N for atom numbering...
added slash to end of path, new path is DATA/LNiH2_library_001_to_download_2026_03_24/YOWGAJZEYGYTFY-SNVBAGLBSA-N/
Processing DAT

,xyz_name,path,ligand_type,inchi,SCF_energy,smiles,H1,Ni,H2,N1,C1,C2,N2,ligand_class,feather_path
0,optimization,DATA/LNiH2_library_001_to_download_2026_03_24/...,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425277,[H]C1=C(F)NC(=C2N[C@@]3([H])c4c([H])c([H])c([H...,1,2,3,12,6,5,4,pyox,DATA/LNiH2_library_001_to_download_2026_03_24/...
1,optimization,DATA/LNiH2_library_001_to_download_2026_03_24/...,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425257,[H]C1=C(F)NC(=C2N[C@@]3([H])c4c([H])c([H])c([H...,1,2,3,12,6,5,4,pyox,DATA/LNiH2_library_001_to_download_2026_03_24/...
2,optimization,DATA/LNiH2_library_001_to_download_2026_03_24/...,core_5,DVLYVQBEMHJQAQ-SFHVURJKSA-N,-2811.494785,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,1,2,3,15,6,5,4,pyox,DATA/LNiH2_library_001_to_download_2026_03_24/...
3,optimization,DATA/LNiH2_library_001_to_download_2026_03_24/...,core_5,DVLYVQBEMHJQAQ-SFHVURJKSA-N,-2811.49527,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,1,2,3,15,6,5,4,pyox,DATA/LNiH2_library_001_to_download_2026_03_24/...
4,optimization,DATA/LNiH2_library_001_to_download_2026_03_24/...,core_5,DVLYVQBEMHJQAQ-SFHVURJKSA-N,-2811.493515,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,1,2,3,15,6,5,4,pyox,DATA/LNiH2_library_001_to_download_2026_03_24/...


# ORCA Descriptors

### Geometric Descriptors

In [6]:
from desc_utils.data_extraction import *
os.chdir(ROOT_DIR)

results_xyz_dict = get_xyz_descriptors(df_num, atoms, bonds, angles, dihedrals, quadrants=None, overwrite_json=False)
df_xyz_desc = pd.DataFrame.from_dict(results_xyz_dict, orient='index')
df_xyz_desc.set_index('path', inplace=True)
df_xyz_desc.head(5)

  0%|          | 0/308 [00:00<?, ?it/s]

Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_25/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_22/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_14/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_13/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_47/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_78/xyz.json.
Loading existing results f

100%|██████████| 308/308 [00:00<00:00, 4264.76it/s]

Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_65/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_91/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_31/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_7/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_36/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_9/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_38/xyz.json.
Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_43/xyz.json.
Loading existing results f

,%Vbur_atom_C1_radius_2.0,%Vbur_atom_C1_radius_2.5,%Vbur_atom_C1_radius_3.0,%Vbur_atom_C1_radius_3.5,%Vbur_atom_C1_radius_4.0,%Vbur_atom_C1_radius_4.5,%Vbur_atom_C1_radius_5.0,%Vbur_atom_C2_radius_2.0,%Vbur_atom_C2_radius_2.5,%Vbur_atom_C2_radius_3.0,...,Sterimol_Ni_H1_L,Sterimol_Ni_H2_Bmax,Sterimol_Ni_H2_Bmin,Sterimol_Ni_H2_L,Sterimol_Ni_N1_Bmax,Sterimol_Ni_N1_Bmin,Sterimol_Ni_N1_L,Sterimol_Ni_N2_Bmax,Sterimol_Ni_N2_Bmin,Sterimol_Ni_N2_L
path,,,,,,,,,,,,,,,,,,,,,
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,96.009814,87.469071,77.479701,67.318533,57.888244,49.658898,41.905411,94.947701,86.817945,77.904313,...,3.680284,8.042405,1.884272,4.356749,8.034602,1.798475,7.319335,6.787812,2.175292,7.610382
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,95.951705,87.402331,77.444316,67.306886,57.882813,49.653200,41.900484,94.921875,86.796783,77.881034,...,3.686991,8.037996,1.873462,4.369356,8.030114,1.792913,7.319373,6.787940,2.159351,7.604794
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_25,95.990444,88.471806,80.327212,71.723001,63.066771,56.134129,50.164572,97.155863,92.179971,85.843452,...,4.673138,9.945725,3.068531,5.070741,10.085789,3.054849,7.816049,7.687240,4.458104,9.467927
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_22,96.003357,87.488605,78.053300,69.102917,60.746152,54.083992,48.315056,95.767691,89.114794,82.334811,...,5.070890,7.630035,4.016237,4.680611,9.838185,2.951335,7.772605,7.659860,4.121607,10.032296
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_14,96.067924,87.843469,79.606488,71.722419,64.021088,57.496092,51.465383,97.443182,92.835981,86.682434,...,4.356220,10.990912,3.077963,3.930265,10.946188,3.216689,8.127822,7.918820,4.238545,11.095262


### Charges and Energy Descriptors

In [7]:
results_charges_dict = get_all_charges(df_num, atoms)
df_charges_desc = pd.DataFrame.from_dict(results_charges_dict, orient='index')
df_charges_desc.set_index('path', inplace=True)
df_charges_desc.head(5)

Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW J

,mulliken_H1,mulliken_H2,mulliken_C1,mulliken_C2,mulliken_N1,mulliken_N2,mulliken_Ni,loewdin_H1,loewdin_H2,loewdin_C1,...,saniso_N1,saniso_N2,saniso_Ni,siso_H1,siso_H2,siso_C1,siso_C2,siso_N1,siso_N2,siso_Ni
path,,,,,,,,,,,,,,,,,,,,,
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,-0.077996,-0.097464,0.109601,0.291640,-0.048309,-0.049192,-0.348348,-0.069781,-0.071689,-0.149184,...,366.483239,237.285020,-1920.025043,39.219221,40.217872,37.668341,17.161352,-2.515718,18.126859,-4049.465402
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,-0.079398,-0.097876,0.108511,0.292047,-0.048006,-0.049183,-0.347179,-0.069868,-0.071959,-0.149172,...,367.025926,240.718564,-1934.564548,39.198500,40.214541,37.658881,17.139805,-2.552599,18.228884,-4060.608625
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_25,-0.098376,-0.074072,0.030477,0.364906,-0.002199,-0.123407,-0.325068,-0.066446,-0.064130,-0.149660,...,397.221237,237.660611,-2606.009812,38.720765,42.552337,36.398284,17.046468,-34.346185,15.348416,-4296.334652
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_22,-0.074343,-0.101332,0.014713,0.342346,0.000563,-0.113657,-0.323956,-0.063714,-0.066555,-0.148204,...,391.974231,233.680584,-2567.533789,42.661792,38.654647,36.125690,17.525786,-34.384262,19.019704,-4349.062139
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_14,-0.080924,-0.090248,-0.008087,0.345838,-0.011580,-0.099585,-0.355787,-0.073015,-0.067048,-0.147685,...,326.209545,255.404107,-2356.927868,38.409917,41.993519,36.788141,18.461544,-35.021918,15.131855,-4358.894092


### NBO Descriptors

In [8]:
results_nbo_dict = get_all_nbo_properties(df_num, atoms, basename="singlepoint")
df_nbo_desc = results_nbo_dict.copy()
df_nbo_desc.set_index("path", inplace=True)
df_nbo_desc.head(5)

# filter homo informations to restrict to the range homo-5 to lumo+5
homo_lumo_range = [1, 2, 3, 4, 5]
colums_to_drop = []
for c in df_nbo_desc.columns:
    if re.search(r'_hm\d+_', c):
        if not any(f'_hm{h}_' in c for h in homo_lumo_range):
            colums_to_drop.append(c)
    elif re.search(r'_lp\d+_', c):
        if not any(f'_lp{l}_' in c for l in homo_lumo_range):
            colums_to_drop.append(c)
print(colums_to_drop)

# remove analysis info:
for c in df_nbo_desc.columns:
    if "analysis_info" in c:
        colums_to_drop.append(c)


# Note: the notation hmX means homo - X and lpX means lumo + X.

df_nbo_desc = df_nbo_desc.drop(columns=colums_to_drop)
df_nbo_desc.head(5)


Could not find any complete NBO section in the file
['nbo_hm78_H1_Ni_type', 'nbo_hm78_H1_Ni_bond_index', 'nbo_hm78_H1_Ni_occupancy', 'nbo_hm78_H1_Ni_energy', 'nbo_hm36_H1_Ni_type', 'nbo_hm36_H1_Ni_bond_index', 'nbo_hm36_H1_Ni_occupancy', 'nbo_hm36_H1_Ni_energy', 'nbo_lp6_H1_type', 'nbo_lp6_H1_bond_index', 'nbo_lp6_H1_occupancy', 'nbo_lp6_H1_energy', 'nbo_lp7_H1_type', 'nbo_lp7_H1_bond_index', 'nbo_lp7_H1_occupancy', 'nbo_lp7_H1_energy', 'nbo_lp8_H1_type', 'nbo_lp8_H1_bond_index', 'nbo_lp8_H1_occupancy', 'nbo_lp8_H1_energy', 'nbo_lp9_H1_type', 'nbo_lp9_H1_bond_index', 'nbo_lp9_H1_occupancy', 'nbo_lp9_H1_energy', 'nbo_lp10_H1_type', 'nbo_lp10_H1_bond_index', 'nbo_lp10_H1_occupancy', 'nbo_lp10_H1_energy', 'nbo_lp41_H2_type', 'nbo_lp41_H2_bond_index', 'nbo_lp41_H2_occupancy', 'nbo_lp41_H2_energy', 'nbo_lp42_H2_type', 'nbo_lp42_H2_bond_index', 'nbo_lp42_H2_occupancy', 'nbo_lp42_H2_energy', 'nbo_lp43_H2_type', 'nbo_lp43_H2_bond_index', 'nbo_lp43_H2_occupancy', 'nbo_lp43_H2_energy', 'nbo_lp44

,npa_H1_charge,npa_H1_core,npa_H1_valence,npa_H1_rydberg,npa_H1_total,npa_H2_charge,npa_H2_core,npa_H2_valence,npa_H2_rydberg,npa_H2_total,...,nho_directionality_hm_H1_Ni_hybrid_theta,nho_directionality_hm_H1_Ni_hybrid_phi,nho_directionality_hm_H1_Ni_deviation,nho_directionality_hm_C2_C1_type,nho_directionality_hm_C2_C1_bond_index,nho_directionality_hm_C2_C1_line_theta,nho_directionality_hm_C2_C1_line_phi,nho_directionality_hm_C2_C1_hybrid_theta,nho_directionality_hm_C2_C1_hybrid_phi,nho_directionality_hm_C2_C1_deviation
path,,,,,,,,,,,,,,,,,,,,,
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,-0.21653,0.0,1.21368,0.00285,1.21653,-0.22963,0.0,1.22667,0.00296,1.22963,...,NaN,NaN,NaN,BD,1.0,111.5,0.2,112.5,357.0,3.2
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,-0.21662,0.0,1.21378,0.00284,1.21662,-0.23047,0.0,1.22752,0.00295,1.23047,...,NaN,NaN,NaN,BD,1.0,110.0,359.7,111.0,356.5,3.2
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_25,-0.21984,0.0,1.21684,0.00300,1.21984,-0.23576,0.0,1.23294,0.00282,1.23576,...,NaN,NaN,NaN,BD,1.0,100.5,337.9,99.3,336.2,2.0
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_22,-0.23509,0.0,1.23230,0.00279,1.23509,-0.22010,0.0,1.21712,0.00298,1.22010,...,NaN,NaN,NaN,BD,1.0,98.2,338.9,97.6,336.6,2.3
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_14,-0.23132,0.0,1.22858,0.00274,1.23132,-0.21785,0.0,1.21489,0.00297,1.21785,...,NaN,NaN,NaN,BD,1.0,90.8,352.9,89.4,350.0,3.2


### Vertical Redox Descriptors

In [9]:
results_redox_dict = get_redox_descriptors(
    df_num,
    atoms,
    descriptorPrefixes=["nbo_charge", "cm5"],
    nbo_block_map={"singlepoint": 1, "oxidation": 2, "reduction": 3},
)
df_redox_desc = pd.DataFrame.from_dict(results_redox_dict, orient='index')
df_redox_desc.set_index('path', inplace=True)
df_redox_desc.head(5)

Could not find any complete NBO section in the file
Could not find any complete NBO section in the file
Could not find any complete NBO section in the file


,Vertical_Ionization_Potential,Vertical_Electron_Affinity,Hardness,Electronegativity,Electrophilicity,fplus_nbo_charge_H1,fminus_nbo_charge_H1,fzero_nbo_charge_H1,fplus_nbo_charge_H2,fminus_nbo_charge_H2,...,fzero_cm5_C2,fplus_cm5_N1,fminus_cm5_N1,fzero_cm5_N1,fplus_cm5_N2,fminus_cm5_N2,fzero_cm5_N2,fplus_cm5_Ni,fminus_cm5_Ni,fzero_cm5_Ni
path,,,,,,,,,,,,,,,,,,,,,
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,-0.208644,-0.036060,-0.086292,-0.122352,-0.086741,0.04749,0.10285,0.075170,0.06720,0.12403,...,0.037715,0.050102,-0.014710,0.017696,0.052991,-0.018316,0.017337,0.077775,0.433711,0.255743
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,-0.208691,-0.036117,-0.086287,-0.122404,-0.086819,0.04748,0.10305,0.075265,0.06723,0.12436,...,0.037726,0.050111,-0.014717,0.017697,0.053060,-0.018285,0.017388,0.077710,0.433477,0.255593
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_25,-0.213338,-0.046233,-0.083552,-0.129785,-0.100800,0.04229,0.09780,0.070045,0.06443,0.11995,...,0.031622,0.050859,-0.013946,0.018457,0.051549,-0.017930,0.016809,0.071710,0.416382,0.244046
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_22,-0.213240,-0.046011,-0.083614,-0.129625,-0.100478,0.06536,0.12069,0.093025,0.04273,0.09803,...,0.032287,0.050951,-0.014019,0.018466,0.051826,-0.018192,0.016817,0.072292,0.416921,0.244606
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_14,-0.212551,-0.045489,-0.083531,-0.129020,-0.099640,0.04525,0.09811,0.071680,0.06723,0.12080,...,0.032841,0.048564,-0.012434,0.018065,0.049053,-0.017822,0.015615,0.074004,0.411836,0.242920


### Summary of Descriptors

In [10]:
# Build a single summary table matched by molecular path
from desc_utils.data_extraction import _truncate_to_orca_descriptors

# 1) Base table (normalize df_num path so it matches extractor outputs)
_repo_root = Path('desc_utils/data_extraction.py').resolve().parent.parent
df_num_idx = df_num.copy()
df_num_idx['path'] = df_num_idx['path'].map(lambda p: _truncate_to_orca_descriptors(p, _repo_root))
df_num_idx = df_num_idx.set_index("path")

# 2) Concatenate all descriptor tables on index (= path)
total_df = pd.concat(
    [df_num_idx, df_xyz_desc, df_charges_desc, df_redox_desc, df_nbo_desc],
    axis=1,
    join="outer"
)

# 3) Remove duplicated column names (keep first occurrence)
total_df = total_df.loc[:, ~total_df.columns.duplicated()]

# 4) Restore path as a normal column
total_df = total_df.reset_index().rename(columns={"index": "path"})

print("total_df shape:", total_df.shape)
total_df.head(3)

total_df shape: (308, 516)


,path,xyz_name,ligand_type,inchi,SCF_energy,smiles,H1,Ni,H2,N1,...,nho_directionality_hm_H1_Ni_hybrid_theta,nho_directionality_hm_H1_Ni_hybrid_phi,nho_directionality_hm_H1_Ni_deviation,nho_directionality_hm_C2_C1_type,nho_directionality_hm_C2_C1_bond_index,nho_directionality_hm_C2_C1_line_theta,nho_directionality_hm_C2_C1_line_phi,nho_directionality_hm_C2_C1_hybrid_theta,nho_directionality_hm_C2_C1_hybrid_phi,nho_directionality_hm_C2_C1_deviation
0,orca_descriptors/DATA/LNiH2_library_001_to_dow...,optimization,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425277,[H]C1=C(F)NC(=C2N[C@@]3([H])c4c([H])c([H])c([H...,1,2,3,12,...,NaN,NaN,NaN,BD,1.0,111.5,0.2,112.5,357.0,3.2
1,orca_descriptors/DATA/LNiH2_library_001_to_dow...,optimization,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425257,[H]C1=C(F)NC(=C2N[C@@]3([H])c4c([H])c([H])c([H...,1,2,3,12,...,NaN,NaN,NaN,BD,1.0,110.0,359.7,111.0,356.5,3.2
2,orca_descriptors/DATA/LNiH2_library_001_to_dow...,optimization,core_5,DVLYVQBEMHJQAQ-SFHVURJKSA-N,-2811.494785,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,1,2,3,15,...,NaN,NaN,NaN,BD,1.0,100.5,337.9,99.3,336.2,2.0


# DescriPyTor

### Convert the ORCA output files to pseudo Gaussian log files for compatibility with descripytor generation. 

This step is necessary because descripy is designed to work with Gaussian log files, and the ORCA output files need to be converted to this format for descripy to process them correctly.

In [11]:
# convert orca files to log files | a log file is created for each conf_* subfolder in the orca_folder, and contains the same information as the orca output file but in a format that can be read by the descriptor calculation code
script_path = Path("orca2gaussian/orca_2_gauss.py")

if not script_path.exists():
    raise FileNotFoundError(f"Script not found: {script_path.resolve()}")
if not orca_folder.exists():
    raise FileNotFoundError(f"Folder not found: {orca_folder.resolve()}")


try:
    subprocess.run(
                [sys.executable, str(script_path), "--orca-folder", str(orca_folder)],
                capture_output=True,
                text=True,
                check=True,
            )

except subprocess.CalledProcessError as exc:
    print(f"Failed for {orca_folder}: {exc.stderr.strip() or exc}")

### DescriPyTor

#### Convert the log files to feather files for use in DescriPyTor.

In [12]:
# Load Descripytor feather file utils
DESCRIPYTOR_DIR = Path("DescriPyTor/MolFeatures")
OVERWRITE_FEATHER = False  # Set to True to overwrite existing feather files, False to skip if feather file already exists
print(f"Setting DESCRIPYTOR_DIR to: {DESCRIPYTOR_DIR}")
sys.path.append(DESCRIPYTOR_DIR)
sys.path.append(os.path.join(DESCRIPYTOR_DIR, 'M3_modeler'))
sys.path.append(os.path.join(DESCRIPYTOR_DIR, 'M2_data_extractor'))
sys.path.append(os.path.join(DESCRIPYTOR_DIR, 'utils'))

try:
    from data_extractor import Molecules, extract_connectivity
    from feather_extractor import logs_to_feather
    import gaussian_handler
    from gaussian_handler import feather_file_handler
    from help_functions import dict_to_horizontal_df, get_df_from_file, pick_samples_to_remove_for_distribution
    import help_functions
    import visualize

    print("All modules imported successfully.")
    
except ModuleNotFoundError as e:
    print(f"Module import failed: {e} \nPlease ensure all required modules are installed and available in the path specified on the right hand corner of the notebook.")


pd.set_option('display.max_columns', None)

# log_dir = os.path.join(examples_path, 'logs_example')

# iterate over all the log files in the orca_folder and check if they exist, print the path of the log file if it exists, otherwise print a warning message, and extract the log files to a feather file.
pwd_before = os.getcwd()
print(f"Current working directory: {os.getcwd()}")
for inchi_folder in os.listdir(orca_folder):
    if os.path.isdir(os.path.join(orca_folder, inchi_folder)):
        os.chdir(os.path.join(orca_folder, inchi_folder))
    else:
        print(f"Warning: {os.path.join(orca_folder, inchi_folder)} is not a directory, skipping.")
        continue
    
    for conf_folder in os.listdir('.'):
        if conf_folder.startswith('conf_') and os.path.isdir(conf_folder):
            os.chdir(conf_folder)
            log_file_path = None
            for f in os.listdir('.'):
                if f.endswith('.log'):
                    log_file_path = os.path.join(os.getcwd(), f)
            if log_file_path is None:
                print(f"No log file found in {os.getcwd()}")
            else:
                feather_file = log_file_path.replace('.log', '.feather')
                if os.path.exists(feather_file) and not OVERWRITE_FEATHER:
                    print(f"Feather file already exists, skipping: {feather_file}")
                else:
                    try:
                        logs_to_feather(os.getcwd(), feather_dir=os.getcwd())
                        print(f"Logs successfully converted to feather format at: {os.getcwd()}")
                    except Exception as e:
                        print(f"Error converting logs in {os.getcwd()} to feather format: {e}")
            os.chdir('..')
        
    os.chdir(pwd_before)
    

Setting DESCRIPYTOR_DIR to: DescriPyTor/MolFeatures
All modules imported successfully.
Current working directory: /Users/julesschleinitz/Desktop/Code/orca_descriptors
Feather file already exists, skipping: /Users/julesschleinitz/Desktop/Code/orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1/conf_1.feather
Feather file already exists, skipping: /Users/julesschleinitz/Desktop/Code/orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2/conf_2.feather
Feather file already exists, skipping: /Users/julesschleinitz/Desktop/Code/orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_25/conf_25.feather
Feather file already exists, skipping: /Users/julesschleinitz/Desktop/Code/orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_22/conf_22.feather
Feather file already exists, skipping: /Users/julesschleinitz/Desktop/Code/or

#### Extract DescriPyTor descriptors (vibrations, dipoles and sterimol descriptors)

In [13]:

os.chdir(ROOT_DIR)

# renumbering of the atoms in the feather files according to the df_num numberings
renumbering_convention = {"H1": 1, "H2": 2, "Ni": 3, "N1": 4, "C1": 5, "C2": 6, "N2": 7}

desc_all = {}
for i, row in tqdm(df_num.iterrows(), total=df_num.shape[0]):
    os.chdir(ROOT_DIR)
    desc_mol = {}
    # read path
    feather_dir = row['path']
    desc_mol['feather_dir'] = feather_dir
    # create Molecule object and renumber atoms according to the df_num numberings
    mols = Molecules(molecules_dir_name=feather_dir)

    # global descriptors
    ## 1-atom descriptors
    charges_dict    = mols.get_charge_df_dict(atom_indices=[row[x] for x in atoms]) # NBO
    index = charges_dict.index[0]
    for j, atom in enumerate(atoms):
        desc_mol[f'charge_NBO_{atom}'] = charges_dict.loc[index, f"nbo_atom_{row[atom]}"]

    ## 2-atom descriptors
    # sterimol descriptors (uses "bonds" list defined above to determine which sterimol descriptors to calculate)
    sterimol_descriptors    = mols.get_sterimol_dict(atom_indices=[[row[x], row[y]] for x, y in bonds])
    for j, (x1, x2) in enumerate(bonds):
        desc_mol[f'B1_{x1}-{x2}']           = sterimol_descriptors.loc[index, f'B1_{row[x1]}-{row[x2]}']
        desc_mol[f'B5_{x1}-{x2}']           = sterimol_descriptors.loc[index, f'B5_{row[x1]}-{row[x2]}']
        desc_mol[f'L_{x1}-{x2}']            = sterimol_descriptors.loc[index, f'L_{row[x1]}-{row[x2]}']
        desc_mol[f'loc_B5_{x1}-{x2}']       = sterimol_descriptors.loc[index, f'loc_B5_{row[x1]}-{row[x2]}']
        desc_mol[f'B1-B5_angle_{x1}-{x2}']  = sterimol_descriptors.loc[index, f'B1_B5_angle_{row[x1]}-{row[x2]}']

    bond_length_dict        = mols.get_bond_length_dict(atom_pairs=[[row[x], row[y]] for x, y in bonds]) # dataframe with bond length descriptors for specified bonds
    for j, (x1, x2) in enumerate(bonds):
        desc_mol[f'dist_{x1}-{x2}'] = bond_length_dict.loc[index, f'0_bond_length_{row[x1]}-{row[x2]}']

    vibrations_dict         = mols.get_stretch_vibration_dict(atom_pairs=[[row[x], row[y]] for x, y in bonds]) # dataframe with vibration descriptors for all molecules in the feather file, can be filtered for specific vibrations of interest based on the atoms involved in the vibration
    for j, (x1, x2) in enumerate(bonds):
        desc_mol[f'vib_freq_{x1}-{x2}'] = vibrations_dict.loc[index, f'Frequency_Stretch_{row[x1]}_{row[x2]}']
        desc_mol[f'vib_amp_{x1}-{x2}'] = vibrations_dict.loc[index, f'Amplitude_Stretch_{row[x1]}_{row[x2]}']

    vibrations_bend_dict    = mols.get_bend_vibration_dict(atom_pairs=[[row[x], row[y]] for x, y in pairs_bending]) # dataframe with bend vibration descriptors for specified angles
    for j, (x1, x2) in enumerate(pairs_bending):
        desc_mol[f'vib_freq_bend_{x1}-{x2}'] = vibrations_bend_dict.loc[index, f'Frequency_Bending_{row[x1]}-{row[x2]}']
        desc_mol[f'vib_amp_bend_{x1}-{x2}']  = vibrations_bend_dict.loc[index, f'Cross_mag_Bending_{row[x1]}-{row[x2]}']
    
    charges_diff_dict       = mols.get_charge_diff_df_dict(atom_indices=[[row[x], row[y]] for x, y in bonds]) # dataframe with charge difference descriptors for specified bonds
    for j, (x1, x2) in enumerate(bonds):
        desc_mol[f'charge_NBO_diff_{x1}-{x2}'] = charges_diff_dict.loc[index, f'nbo_diff_{row[x1]}-{row[x2]}']

    ## 3-atom descriptors
    dipole_ni = mols.get_dipole_dict(atom_indices=[row["Ni"], row["N2"], row["N1"]]) # dataframe with dipole moment descriptors for specified atom
    desc_mol['dipole'] = dipole_ni.loc[index, f'total_dipole_{row["Ni"]}-{row["N2"]}-{row["N1"]}']
    desc_mol['dipole_x_Ni-N2-N1'] = dipole_ni.loc[index, f'dipole_x_{row["Ni"]}-{row["N2"]}-{row["N1"]}']
    desc_mol['dipole_y_Ni-N2-N1'] = dipole_ni.loc[index, f'dipole_y_{row["Ni"]}-{row["N2"]}-{row["N1"]}']
    desc_mol['dipole_z_Ni-N2-N1'] = dipole_ni.loc[index, f'dipole_z_{row["Ni"]}-{row["N2"]}-{row["N1"]}']

    angles_dict = mols.get_bond_angle_dict(atom_indices=[[row[x], row[y], row[z]] for x, y, z in angles])
    for j, (x1, x2, x3) in enumerate(angles):
        desc_mol[f'angle_{x1}-{x2}-{x3}'] = angles_dict.loc[index, f'0_angle_[{row[x1]}, {row[x2]}, {row[x3]}]']

    desc_all[i] = desc_mol

df_desc = pd.DataFrame.from_dict(desc_all, orient='index')
df_desc.head(5)

  0%|          | 0/308 [00:00<?, ?it/s]

Molecules Loaded: ['conf_1.feather'] Failed Molecules: []


  0%|          | 1/308 [00:00<04:19,  1.18it/s]

Calculating stretch vibration for atoms [[6, 12], [12, 6], [5, 4], [4, 5], [2, 12], [2, 4], [2, 1], [2, 3], [6, 5], [5, 6]] with threshold 1400 
 Remember : ALWAYS LOOK AT THE RESULTING VIBRATION
Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_1 for [2, 12] 

Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_1 for [2, 4] 

Molecules Loaded: ['conf_2.feather'] Failed Molecules: []


  1%|          | 2/308 [00:01<04:04,  1.25it/s]

...

100%|█████████▉| 307/308 [07:14<00:00,  1.03it/s]

Calculating stretch vibration for atoms [[6, 16], [16, 6], [5, 4], [4, 5], [2, 16], [2, 4], [2, 1], [2, 3], [6, 5], [5, 6]] with threshold 1400 
 Remember : ALWAYS LOOK AT THE RESULTING VIBRATION
Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_1 for [2, 16] 

Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_1 for [2, 4] 

Molecules Loaded: ['conf_2.feather'] Failed Molecules: []


100%|██████████| 308/308 [07:14<00:00,  1.41s/it]

Calculating stretch vibration for atoms [[6, 16], [16, 6], [5, 4], [4, 5], [2, 16], [2, 4], [2, 1], [2, 3], [6, 5], [5, 6]] with threshold 1400 
 Remember : ALWAYS LOOK AT THE RESULTING VIBRATION
Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_2 for [2, 16] 

Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_2 for [2, 4] 



,feather_dir,charge_NBO_H1,charge_NBO_H2,charge_NBO_C1,charge_NBO_C2,charge_NBO_N1,charge_NBO_N2,charge_NBO_Ni,B1_C1-N1,B5_C1-N1,L_C1-N1,loc_B5_C1-N1,B1-B5_angle_C1-N1,B1_N1-C1,B5_N1-C1,L_N1-C1,loc_B5_N1-C1,B1-B5_angle_N1-C1,B1_C2-N2,B5_C2-N2,L_C2-N2,loc_B5_C2-N2,B1-B5_angle_C2-N2,B1_N2-C2,B5_N2-C2,L_N2-C2,loc_B5_N2-C2,B1-B5_angle_N2-C2,B1_Ni-N1,B5_Ni-N1,L_Ni-N1,loc_B5_Ni-N1,B1-B5_angle_Ni-N1,B1_Ni-N2,B5_Ni-N2,L_Ni-N2,loc_B5_Ni-N2,B1-B5_angle_Ni-N2,B1_Ni-H1,B5_Ni-H1,L_Ni-H1,loc_B5_Ni-H1,B1-B5_angle_Ni-H1,B1_Ni-H2,B5_Ni-H2,L_Ni-H2,loc_B5_Ni-H2,B1-B5_angle_Ni-H2,B1_C1-C2,B5_C1-C2,L_C1-C2,loc_B5_C1-C2,B1-B5_angle_C1-C2,B1_C2-C1,B5_C2-C1,L_C2-C1,loc_B5_C2-C1,B1-B5_angle_C2-C1,dist_C1-N1,dist_N1-C1,dist_C2-N2,dist_N2-C2,dist_Ni-N1,dist_Ni-N2,dist_Ni-H1,dist_Ni-H2,dist_C1-C2,dist_C2-C1,vib_freq_C1-N1,vib_amp_C1-N1,vib_freq_N1-C1,vib_amp_N1-C1,vib_freq_C2-N2,vib_amp_C2-N2,vib_freq_N2-C2,vib_amp_N2-C2,vib_freq_Ni-N1,vib_amp_Ni-N1,vib_freq_Ni-N2,vib_amp_Ni-N2,vib_freq_Ni-H1,vib_amp_Ni-H1,vib_freq_Ni-H2,vib_amp_Ni-H2,vib_freq_C1-C2,vib_amp_C1-C2,vib_freq_C2-C1,vib_amp_C2-C1,vib_freq_bend_H1-H2,vib_amp_bend_H1-H2,vib_freq_bend_N1-C2,vib_amp_bend_N1-C2,vib_freq_bend_N2-C1,vib_amp_bend_N2-C1,charge_NBO_diff_C1-N1,charge_NBO_diff_N1-C1,charge_NBO_diff_C2-N2,charge_NBO_diff_N2-C2,charge_NBO_diff_Ni-N1,charge_NBO_diff_Ni-N2,charge_NBO_diff_Ni-H1,charge_NBO_diff_Ni-H2,charge_NBO_diff_C1-C2,charge_NBO_diff_C2-C1,dipole,dipole_x_Ni-N2-N1,dipole_y_Ni-N2-N1,dipole_z_Ni-N2-N1,angle_H2-Ni-H1,angle_Ni-N1-C1,angle_Ni-N2-C2,angle_N1-C1-C2,angle_N2-C2-C1,angle_N1-Ni-N2,angle_C1-Ni-C2
0,DATA/LNiH2_library_001_to_download_2026_03_24/...,-0.21653,-0.22963,0.11686,0.50297,-0.46491,-0.47278,0.37908,1.6991,4.3120,4.6065,1.8397,179.9947,1.8611,8.4536,5.8585,2.4020,136.5252,1.6991,6.8659,5.3416,2.8889,124.7608,1.9308,6.8659,5.9723,-1.6114,151.5772,2.0079,7.9342,6.8193,-0.4638,140.4172,2.0809,6.6878,7.1104,1.3597,85.3772,1.919,1.9201,2.4654,0.0,34.5459,1.919,1.9201,2.4624,0.0,34.5459,1.8854,6.1346,6.7984,5.5038,159.5956,1.6998,3.9380,6.2132,3.7593,90.0509,1.353525,1.353525,1.277524,1.277524,1.997576,1.936436,1.465408,1.462377,1.455706,1.455706,1428.29,0.0,1428.29,0.0,1428.29,0.0,1428.29,0.0,NaN,NaN,NaN,NaN,1428.29,0.0,1428.29,0.0,1428.29,0.0,1428.29,0.0,1310.99,0.0,1310.99,0.0,1310.99,0.0,0.58177,-0.58177,0.97575,-0.97575,0.84399,0.85186,0.59561,0.60871,-0.38611,0.38611,3.749951,-0.222724,3.640507,0.871343,80.285471,112.981322,113.778353,111.707045,119.327386,82.181594,30.439763
1,DATA/LNiH2_library_001_to_download_2026_03_24/...,-0.21662,-0.23047,0.11683,0.50311,-0.46484,-0.47284,0.37963,1.6991,4.3119,4.6064,1.8398,179.9601,1.8636,8.4487,5.8557,2.4083,136.5769,1.6991,6.8727,5.3207,2.8673,124.8268,1.9325,6.8727,5.9731,-1.5897,152.5569,2.0133,7.9298,6.8194,-0.4775,140.3940,2.0665,6.6878,7.1048,4.0966,86.0972,1.919,1.9201,2.4659,0.0,34.5459,1.919,1.9201,2.4633,0.0,34.5459,1.8811,6.1351,6.7936,5.4978,159.8085,1.6998,3.9376,6.2134,3.7598,89.9782,1.353520,1.353520,1.277506,1.277506,1.997321,1.936877,1.465891,1.463258,1.455765,1.455765,1428.32,0.0,1428.32,0.0,1428.32,0.0,1428.32,0.0,NaN,NaN,NaN,NaN,1428.32,0.0,1428.32,0.0,1428.32,0.0,1428.32,0.0,1310.99,0.0,1310.99,0.0,1310.99,0.0,0.58167,-0.58167,0.97595,-0.97595,0.84447,0.85247,0.59625,0.61010,-0.38628,0.38628,3.744847,-0.222501,3.635383,0.870840,80.423241,112.983518,113.755704,111.715783,119.338751,82.195500,30.442192
2,DATA/LNiH2_library_001_to_download_2026_03_24/...,-0.21984,-0.23576,0.11566,0.51649,-0.42570,-0.49418,0.39430,1.6991,5.9192,4.6873,1.6383,166.6662,4.0773,7.9415,7.7080,2.1073,117.5082,1.6991,8.0730,6.1007,-0.8508,144.1682,4.0843,8.0728,6.0448,2.1300,68.8488,3.9375,9.9856,7.2960,1.2475,112.0676,4.9938,7.5671,8.9679,-1.3111,166.6627,1.919,1.9201,2.4620,0.0,34.5459,1.919,1.9201,2.4608,0.0,34.5459,3.9697,7.2447,7.6971,2.1121,69.2596,2.2372,3.3991,7.5498,3.6594,68.0624,1.342071,1.342071,1.279223,1.279223,1.952133,1.988616,1.461994,1.460825,1.457091,1.457091,1409.66,0.0,1409.66,0.0,1409.66,0.0,1409.66,0.0

# Combining ORCA and DescriPyTor descriptors

In [14]:
import re

# --- 1) Build merge key (conf folder) for total_df ---
total_work = total_df.drop(columns=atoms).copy()

def _normalize_conf_dir(p):
    if pd.isna(p):
        return np.nan
    p = str(p).replace("\\", "/")
    if p.endswith(".feather"):
        p = os.path.dirname(p)
    p = p.rstrip("/")
    p = re.sub(r"^orca_descriptors/", "", p)  # align with df_desc paths
    return p

total_work["conf_dir"] = total_work["path"].map(_normalize_conf_dir)

# --- 2) Build merge key for df_desc ---
desc_work = df_desc.copy()
if "feather_dir" not in desc_work.columns:
    raise KeyError("df_desc must contain 'feather_dir'.")

desc_work["conf_dir"] = desc_work["feather_dir"].map(_normalize_conf_dir)

# --- 3) Combine descriptor tables ---
combined_df = total_work.merge(
    desc_work,
    on="conf_dir",
    how="outer",
    suffixes=("_total", "_desc")
)

# If duplicate energy columns exist, keep one unified SCF_energy
if "SCF_energy" not in combined_df.columns:
    scf_candidates = [c for c in combined_df.columns if c.startswith("SCF_energy")]
    if scf_candidates:
        combined_df["SCF_energy"] = combined_df[scf_candidates].bfill(axis=1).iloc[:, 0]

os.chdir(ROOT_DIR)
combined_df.to_csv("combined_descriptors.csv", index=False)

# --- 4) Group key = inchi folder (path without /conf_XX) ---
combined_df["inchi_folder"] = combined_df["conf_dir"].str.replace(
    r"/conf_\d+$", "", regex=True
)

# --- 5) Boltzmann aggregation per inchi folder ---
T = 298.15
kB_hartree_per_K = 3.166811563e-6  # Hartree/K

if "SCF_energy" not in combined_df.columns:
    raise KeyError("SCF_energy column not found after merge.")

numeric_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()

rows = []
for inchi_folder, g in combined_df.groupby("inchi_folder", dropna=True):
    g = g.copy()
    e = pd.to_numeric(g["SCF_energy"], errors="coerce")
    if e.notna().sum() == 0:
        continue

    e_min = e.min()
    idx_low = e.idxmin()

    dE = e - e_min
    w = np.exp(-(dE / (kB_hartree_per_K * T)))
    w = w / np.nansum(w)

    out = {
        "inchi_folder": inchi_folder,
        "inchi": os.path.basename(inchi_folder),
        "min_energy_value": e_min,
        "smiles": g["smiles"].iloc[0] if "smiles" in g.columns else np.nan,
        "ligand_class": g["ligand_class"].iloc[0] if "ligand_class" in g.columns else np.nan
    }

    for col in numeric_cols:
        v = pd.to_numeric(g[col], errors="coerce")

        out[f"{col}_min"] = v.min()
        out[f"{col}_max"] = v.max()
        out[f"{col}_lowE"] = g.loc[idx_low, col] if idx_low in g.index else np.nan

        valid = v.notna() & w.notna()
        out[f"{col}_boltz"] = (
            np.average(v[valid], weights=w[valid]) if valid.any() else np.nan
        )

    rows.append(out)

df_boltzmann_summary = pd.DataFrame(rows).sort_values("inchi_folder").reset_index(drop=True)
df_boltzmann_summary.drop(columns=["min_energy_value"], inplace=True)

print("combined_df shape:", combined_df.shape)
print("df_boltzmann_summary shape:", df_boltzmann_summary.shape)
df_boltzmann_summary.head()

combined_df shape: (308, 626)
df_boltzmann_summary shape: (38, 2396)


inchi_folder  \
0  DATA/LNiH2_library_001_to_download_2026_03_24/...   
1  DATA/LNiH2_library_001_to_download_2026_03_24/...   
2  DATA/LNiH2_library_001_to_download_2026_03_24/...   
3  DATA/LNiH2_library_001_to_download_2026_03_24/...   
4  DATA/LNiH2_library_001_to_download_2026_03_24/...   

                         inchi  \
0  AATATIAROAAKIN-SNVBAGLBSA-N   
1  CCRBBXHXJMPAHM-QGZVFWFLSA-N   
2  CTDSXZLFAXPRHX-MRVPVSSYSA-N   
3  DERMNSMASFMDOJ-AWEZNQCLSA-N   
4  DSZOIEZWGDCKLK-KGLIPLIRSA-N   

                                              smiles ligand_class  \
0  [H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...         pyox   
1  [H]C1=C(OC([H])([H])c2c([H])c([H])c([H])c([H])...         pyox   
2  [H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...         pyox   
3  [H]C1=C([H])C(=C2N[C@@]([H])(C([H])([H])c3c([H...         pyox   
4  [H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]3([H]...         pyox   

   %Vbur_atom_C1_radius_2.0_min  %Vbur_atom_C1_radius_2.0_max  \
0                     96.019499                     96.064695   
1                     95.980759                     96.071152   
2                     95.971074                     96.022727   
3                     95.974303                     96.048554   
4                     96.013042                     96.051782   

   %Vbur_atom_C1_radius_2.0_lowE  %Vbur_atom_C1_radius_2.0_boltz  \
0                      96.019499                       96.041779   
1                      95.980759                       96.018933   
2                      96.003357                       96.009860   
3                      96.003357                       96.018454   
4                      96.013042                       96.032164   

   %Vbur_atom_C1_radius_2.5_min  %Vbur_atom_C1_radius_2.5_max  \
0                     87.552090                     87.583019   
1                     87.390936                     87.488605   
2                     87.390936                     87.434887   
3                     87.513023                     88.419716   
4                     87.400703                     87.400703   

   %Vbur_atom_C1_radius_2.5_lowE  %Vbur_atom_C1_radius_2.5_boltz  \
0                      87.552090                       87.567337   
1                      87.421865                       87.433749   
2                      87.434887                       87.430730   
3                      87.513023                       87.759958   
4                      87.400703                       87.400703   

   %Vbur_atom_C1_radius_3.0_min  %Vbur_atom_C1_radius_3.0_max  \
0                     78.491880                     78.510504   
1                     77.448972                     78.349412   
2                     77.442454                     77.610064   
3                     77.586785                     80.319763   
4                     77.472251                     77.475045   

   %Vbur_atom_C1_radius_3.0_lowE  %Vbur_atom_C1_radius_3.0_boltz  \
0                      78.510504                       78.501323   
1                      78.133381                       78.086878   
2                      77.462940                       77.468252   
3                      77.586785                       78.340031   
4                      77.475045                       77.473666   

   %Vbur_atom_C1_radius_3.5_min  %Vbur_atom_C1_radius_3.5_max  \
0                     69.134947                     69.152418   
1                     67.415207                     69.106993   
2                     67.260879                     67.744829   
3                     67.546823                     71.854035   
4                     67.313874                     67.355223   

   %Vbur_atom_C1_radius_3.5_lowE  %Vbur_atom_C1_radius_3.5_boltz  \
0                      69.152418                       69.143805   
1                      68.678368                       68.609095   
2                      67.260879                       67.312738   
3                   

In [15]:
ROOT_DIR

'/Users/julesschleinitz/Desktop/Code/orca_descriptors'

### Remove noisy descriptors

In [16]:
import importlib
import desc_utils.data_extraction as de

importlib.reload(de)

from desc_utils.data_extraction import filter_numeric_descriptors

df_num_desc_filtered, report = filter_numeric_descriptors(
    df_boltzmann_summary,
    remove_low_occupancy_nbo=True,
    occupancy_boltz_threshold=0.001,
    corr_threshold=0.95,
    corr_pruning_mode="bridge_star_centers", #"component_ranked",
    ratio_threshold=0.01,
    orbital_filter="remove rydbergs",
    cor_orbital_filter="remove cor",
    ni_orbital_filter="remove 4f rydbergs",    # new filter
    nmr_filter="remove aniso",
    bond_analysis_filter="remove phi theta",
)

print("Initial numeric descriptors:", report["initial_numeric_count"])
print("Removed (NBO low occupancy orbitals):", len(report["removed_low_occ_nbo_orbitals"]))
print("Removed (NBO low occupancy columns):", len(report["removed_low_occ_nbo_columns"]))
print("Removed (Rydberg, except Ni):", len(report["removed_rydberg"]))
print("Removed (Ni 4f Rydbergs):", len(report["removed_ni_4f_rydberg"]))
print("Removed (NMR aniso):", len(report["removed_nmr_aniso"]))
print("Removed (phi/theta bond analysis):", len(report["removed_phi_theta"]))
print("Removed (zero std):", len(report["removed_zero_std"]))
print("Removed (std/abs(mean) < 1%):", len(report["removed_low_ratio"]))
print("Removed (high correlation):", len(report["removed_high_corr"]))
print("Final numeric descriptors:", report["final_numeric_count"])

# filter_numeric_descriptors returns only numeric columns, so re-attach meta columns from df_boltzmann_summary
meta_cols = ["smiles", "inchi", "inchi_folder", "ligand_class"]
for col in meta_cols:
    if col in df_boltzmann_summary.columns:
        df_num_desc_filtered[col] = df_boltzmann_summary[col].values

df_num_desc_filtered.head()


Initial numeric descriptors: 2392
Removed (NBO low occupancy orbitals): 96
Removed (NBO low occupancy columns): 400
Removed (Rydberg, except Ni): 168
Removed (Ni 4f Rydbergs): 0
Removed (NMR aniso): 28
Removed (phi/theta bond analysis): 68
Removed (zero std): 132
Removed (std/abs(mean) < 1%): 335
Removed (high correlation): 650
Final numeric descriptors: 559


,%Vbur_atom_C1_radius_3.0_boltz,%Vbur_atom_C1_radius_4.5_max,%Vbur_atom_C1_radius_5.0_lowE,%Vbur_atom_C1_radius_5.0_boltz,%Vbur_atom_C2_radius_2.5_min,%Vbur_atom_C2_radius_3.5_max,%Vbur_atom_C2_radius_3.5_lowE,%Vbur_atom_C2_radius_3.5_boltz,%Vbur_atom_C2_radius_5.0_boltz,%Vbur_atom_H1_radius_2.0_min,%Vbur_atom_H1_radius_2.0_max,%Vbur_atom_H1_radius_2.0_lowE,%Vbur_atom_H1_radius_2.0_boltz,%Vbur_atom_H1_radius_2.5_min,%Vbur_atom_H1_radius_2.5_max,%Vbur_atom_H1_radius_2.5_lowE,%Vbur_atom_H1_radius_2.5_boltz,%Vbur_atom_H1_radius_3.0_min,%Vbur_atom_H1_radius_3.0_lowE,%Vbur_atom_H1_radius_3.0_boltz,%Vbur_atom_H1_radius_4.0_max,%Vbur_atom_H1_radius_5.0_min,%Vbur_atom_H1_radius_5.0_max,%Vbur_atom_H2_radius_2.0_min,%Vbur_atom_H2_radius_2.0_max,%Vbur_atom_H2_radius_2.0_boltz,%Vbur_atom_H2_radius_2.5_min,%Vbur_atom_H2_radius_2.5_max,%Vbur_atom_H2_radius_2.5_lowE,%Vbur_atom_H2_radius_2.5_boltz,%Vbur_atom_H2_radius_4.0_max,%Vbur_atom_H2_radius_5.0_min,%Vbur_atom_H2_radius_5.0_max,%Vbur_atom_H2_radius_5.0_lowE,%Vbur_atom_H2_radius_5.0_boltz,%Vbur_atom_N1_radius_3.0_min,%Vbur_atom_N1_radius_3.0_max,%Vbur_atom_N1_radius_3.0_boltz,%Vbur_atom_N1_radius_3.5_lowE,%Vbur_atom_N1_radius_4.0_min,%Vbur_atom_N1_radius_4.0_max,%Vbur_atom_N1_radius_4.0_lowE,%Vbur_atom_N1_radius_4.0_boltz,%Vbur_atom_N1_radius_4.5_min,%Vbur_atom_N1_radius_4.5_max,%Vbur_atom_N1_radius_4.5_lowE,%Vbur_atom_N1_radius_4.5_boltz,%Vbur_atom_N1_radius_5.0_min,%Vbur_atom_N1_radius_5.0_max,%Vbur_atom_N1_radius_5.0_lowE,%Vbur_atom_N1_radius_5.0_boltz,%Vbur_atom_N2_radius_2.5_min,%Vbur_atom_N2_radius_3.0_boltz,%Vbur_atom_N2_radius_4.0_max,%Vbur_atom_N2_radius_4.5_boltz,%Vbur_atom_Ni_radius_2.0_max,%Vbur_atom_Ni_radius_2.5_min,%Vbur_atom_Ni_radius_2.5_boltz,%Vbur_atom_Ni_radius_4.0_min,%Vbur_atom_Ni_radius_5.0_min,%Vbur_atom_Ni_radius_5.0_lowE,%Vbur_atom_Ni_radius_5.0_boltz,Angle_Ni_N2_C2_min,Dihed_C1_C2_N2_Ni_min,Dihed_C1_C2_N2_Ni_max,Dihed_C1_C2_N2_Ni_lowE,Dihed_C1_C2_N2_Ni_boltz,Dihed_C2_C1_N1_Ni_min,Dihed_C2_C1_N1_Ni_max,Dihed_C2_C1_N1_Ni_lowE,Dihed_C2_C1_N1_Ni_boltz,Dihed_N1_C1_C2_N2_min,Dihed_N1_C1_C2_N2_max,Dihed_N1_C1_C2_N2_boltz,Pyr_angle_C1_max,Pyr_angle_C1_boltz,Pyr_angle_C2_min,Pyr_angle_C2_max,Pyr_angle_C2_lowE,Pyr_angle_C2_boltz,Pyr_angle_N1_min,Pyr_angle_N2_min,Pyr_angle_N2_max,Pyr_angle_N2_lowE,Pyr_angle_N2_boltz,SASA_C1_min,SASA_C1_max,SASA_C1_lowE,SASA_C1_boltz,SASA_C2_min,SASA_C2_max,SASA_C2_boltz,SASA_H1_min,SASA_H2_min,SASA_H2_lowE,SASA_H2_boltz,SASA_N1_min,SASA_N1_max,SASA_N1_lowE,SASA_N1_boltz,SASA_N2_min,SASA_N2_max,SASA_N2_boltz,SASA_Ni_min,SASA_Ni_max,SASA_Ni_lowE,SASA_Ni_boltz,Sterimol_C1_C2_Bmax_max,Sterimol_C1_C2_Bmax_boltz,Sterimol_C1_C2_Bmin_min,Sterimol_C1_C2_L_min,Sterimol_C1_N1_Bmax_max,Sterimol_C1_N1_Bmax_lowE,Sterimol_C1_N1_Bmin_max,Sterimol_C1_N1_L_min,Sterimol_C1_N1_L_max,Sterimol_C2_C1_Bmax_min,Sterimol_C2_C1_Bmax_lowE,Sterimol_C2_C1_Bmin_max,Sterimol_C2_C1_Bmin_boltz,Sterimol_C2_N2_L_max,Sterimol_N1_C1_Bmax_min,Sterimol_N1_C1_L_boltz,Sterimol_N2_C2_Bmin_boltz,Sterimol_N2_C2_L_min,Sterimol_N2_C2_L_boltz,Sterimol_Ni_H1_Bmax_min,Sterimol_Ni_H1_Bmax_max,Sterimol_Ni_H1_Bmax_lowE,Sterimol_Ni_H1_Bmax_boltz,Sterimol_Ni_H1_Bmin_min,Sterimol_Ni_H1_Bmin_max,Sterimol_Ni_H1_Bmin_lowE,Sterimol_Ni_H1_Bmin_boltz,Sterimol_Ni_H1_L_min,Sterimol_Ni_H1_L_max,Sterimol_Ni_H1_L_lowE,Sterimol_Ni_H1_L_boltz,Sterimol_Ni_H2_Bmax_min,Sterimol_Ni_H2_Bmax_max,Sterimol_Ni_H2_Bmax_lowE,Sterimol_Ni_H2_Bmax_boltz,Sterimol_Ni_H2_Bmin_min,Sterimol_Ni_H2_Bmin_max,Sterimol_Ni_H2_Bmin_boltz,Sterimol_Ni_H2_L_min,Sterimol_Ni_H2_L_max,Sterimol_Ni_H2_L_lowE,Sterimol_Ni_H2_L_boltz,Sterimol_Ni_N1_Bmax_min,Sterimol_Ni_N1_Bmax_boltz,Sterimol_Ni_N1_Bmin_min,Sterimol_Ni_N1_Bmin_lowE,Sterimol_Ni_N1_Bmin_boltz,Sterimol_Ni_N1_L_min,Sterimol_Ni_N1_L_boltz,Sterimol_Ni_N2_Bmax_min,Sterimol_Ni_N2_Bmax_lowE,Sterimol_Ni_N2_L_min,mulliken_H1_min,mulliken_H1_max,mulliken_H1_lowE,mulliken_H1_boltz,mulliken_H2_min,mulliken_H2_max,mulliken_H2_lowE,mulliken_H2_boltz,mulliken_C1_boltz,mulliken_C2_boltz,mulliken

In [ ]:
# save to csv
from datetime import datetime
output_csv_path = datetime.now().strftime("%Y%m%d") + "_" + src_dir.split('/')[-1] + "_descriptors" + ".csv"
df_num_desc_filtered.to_csv(output_csv_path, index=False)